# Road Damage Detection — YOLO Training and Experiments


**This notebook documents YOLO training and experiments for detecting four types of road damage in the RDD2022 dataset. It compares models on the validation split and explains why YOLO26s at 640 pixels was selected for the demo.**

**The experiments were run in Google Colab, with the dataset archive and model checkpoints stored in Google Drive.**

## Google Drive Setup
Mount Drive to access the dataset archive and save training results.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

from pathlib import Path

drive_project = Path("/content/drive/MyDrive/road-damage-detection")
drive_project.mkdir(parents=True, exist_ok=True)

print("Project folder ready:", drive_project)

Project folder ready: /content/drive/MyDrive/road-damage-detection


In [3]:
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/road-damage-detection/main_split.zip")

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print("Size:", round(zip_path.stat().st_size / 1024**3, 2), "GiB")

ZIP exists: True
Size: 9.9 GiB


In [4]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Available storage: {free / 1024**3:.1f} GiB")

Available storage: 107.6 GiB


## Dataset Preparation
Extract the dataset locally in Colab and verify the train/val/test file counts.

In [5]:

import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/road-damage-detection/main_split.zip")
extract_path = Path("/content/road-damage-detection/data/processed")

extract_path.mkdir(parents=True, exist_ok=True)

print("Extracting dataset to", extract_path, "...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extraction completed!")

Extracting dataset to /content/road-damage-detection/data/processed ...
Extraction completed!


Verifying the extraction

In [6]:
from pathlib import Path

# Base directory where the dataset was extracted
base_dir = Path("/content/road-damage-detection/data/processed/main_split")

# Define the paths to check
paths_to_check = {
    "Images Train": base_dir / "images/train",
    "Images Val": base_dir / "images/val",
    "Images Test": base_dir / "images/test",
    "Labels Train": base_dir / "labels/train",
    "Labels Val": base_dir / "labels/val",
    "Labels Test": base_dir / "labels/test"
}

# Count and print the number of files in each directory
for label, path in paths_to_check.items():
    if path.exists():
        file_count = len([f for f in path.iterdir() if f.is_file()])
        print(f"{label}: {file_count} files")
    else:
        print(f"{label} directory does not exist at {path}")

Images Train: 30704 files
Images Val: 3835 files
Images Test: 3846 files
Labels Train: 30704 files
Labels Val: 3835 files
Labels Test: 3846 files


## Project Setup
Clone the GitHub repository, organize the dataset, and install the required libraries.

In [7]:
!git clone https://github.com/Aaroh22/road-damage-detection.git /content/road-damage-code

Cloning into '/content/road-damage-code'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 148 (delta 77), reused 107 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.51 MiB | 6.58 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [8]:
from pathlib import Path
import shutil

source = Path("/content/road-damage-detection/data")
destination = Path("/content/road-damage-code/data")

if destination.exists():
    print("Destination already exists. No files moved.")
elif not source.is_dir():
    print("Source folder not found. No files moved.")
else:
    shutil.move(str(source), str(destination))
    print("Dataset moved successfully!")

print("Dataset ready:", (destination / "processed/main_split/images/train").is_dir())

Dataset moved successfully!
Dataset ready: True


In [9]:
from pathlib import Path

project = Path("/content/road-damage-code")

print("GitHub code:", (project / "src/train.py").exists())
print("Dataset:", (project / "data/processed/main_split/images/train").exists())
print("Data config:", (project / "data.yaml").exists())

GitHub code: True
Dataset: True
Data config: True


In [10]:
#installing ultralytics
%pip install -q ultralytics==8.4.204 pyyaml

In [11]:
from ultralytics import YOLO
import yaml

print("Ultralytics and PyYAML are working!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics and PyYAML are working!


In [12]:
import sys

project_path = "/content/road-damage-code"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from src.train import prepare_data_yaml

print("Project imports ready!")

Project imports ready!


## Checkpoint Storage
Link YOLO's output directory to Google Drive to preserve training results.

In [13]:
from pathlib import Path

runs_dir = Path("/content/drive/MyDrive/road-damage-detection/training_runs")

print("Training folder exists:", runs_dir.is_dir())

Training folder exists: True


In [14]:
from pathlib import Path

local_runs = Path("/content/road-damage-code/runs")
drive_runs = Path("/content/drive/MyDrive/road-damage-detection/training_runs")

if not drive_runs.is_dir():
    raise FileNotFoundError("Google Drive training folder not found")

if local_runs.is_symlink():
    if local_runs.resolve() != drive_runs.resolve():
        raise RuntimeError("runs already points to a different location")
    print("Shortcut already exists.")

elif local_runs.exists():
    raise FileExistsError("runs already exists. Check its contents first.")

else:
    local_runs.symlink_to(drive_runs, target_is_directory=True)
    print("Shortcut created successfully!")

print("YOLO outputs will go to:", local_runs.resolve())

Shortcut created successfully!
YOLO outputs will go to: /content/drive/MyDrive/road-damage-detection/training_runs


## Environment Verification
Verify CUDA availability, GPU access, and required project files.

In [15]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU runtime — GPU not required yet.")

CUDA available: False
CPU runtime — GPU not required yet.


In [16]:
from pathlib import Path

project = Path("/content/road-damage-code")
dataset = project / "data/processed/main_split"

print("GitHub code:", (project / "src/train.py").is_file())
print("Dataset:", (dataset / "images/train").is_dir())
print("Data config:", (project / "data.yaml").is_file())
print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Checkpoint shortcut:", (project / "runs").is_symlink())

GitHub code: True
Dataset: True
Data config: True
Drive mounted: True
Checkpoint shortcut: True


## YOLO26s Smoke Test

Train for 1 epoch on 1% of the dataset to verify the training pipeline.
This is a technical validation test, not the final baseline model.

In [17]:
if False:

    import sys
    from ultralytics import YOLO

    sys.path.insert(0, "/content/road-damage-code")
    from src.train import prepare_data_yaml

    # Generate the correct dataset configuration
    data_yaml = prepare_data_yaml()

    # Load pretrained YOLO26s
    model = YOLO("yolo26s.pt")

    # Short test before full training
    model.train(
        data=str(data_yaml),
        epochs=1,
        fraction=0.01,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        val=False,
        seed=42,
        project="/content/road-damage-code/runs",
        name="YOLO26s_smoke_test",
    )

The smoke test successfully verified model initialization, dataset loading, GPU training, validation, and checkpoint saving.


## Experiment 1: YOLO26s Baseline Model

### YOLO26s Full-Dataset Training Benchmark Test

A one-epoch benchmark test is performed on the complete dataset
to estimate GPU memory usage and training time on NVIDIA Tesla T4 GPU before full training.


This benchmark test is only for measuring training feasibility and cost.
It is not intended to produce the final *YOLO26s* model.

In [18]:
if False:

    import time
    from ultralytics import YOLO
    from src.train import prepare_data_yaml

    data_yaml = prepare_data_yaml()

    model = YOLO("yolo26s.pt")

    start = time.time()

    model.train(
        data=str(data_yaml),
        epochs=1,
        fraction=1.0,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        seed=42,
        val=False,
        plots=False,
        close_mosaic=0,
        project="/content/road-damage-code/runs",
        name="YOLO26s_full_dataset_benchmark",
    )

    elapsed = time.time() - start

    print(f"Total benchmark time: {elapsed / 60:.2f} minutes")

### Benchmark Test Results

- **Total runtime:** 15.89 minutes
- **GPU memory:** 2.43 GB

The benchmark completed successfully on the Tesla T4. A 50-epoch run may require approximately 11–14 hours or more, so checkpointing, training resumption and switching to a more powerful GPU will be important.

### YOLO26s (baseline model) Full Training
Train YOLO26s for 50 epochs using our existing training script.

### Configuration

- **Model:** Pretrained YOLO26s
- **Dataset:** RDD2022 (30,704 train / 3,835 val)
- **Epochs:** 50
- **Image size:** 640 × 640
- **Optimizer:** auto
- **Augmentation:** Ultralytics defaults
- **Batch size:** 8
- **GPU:** NVIDIA L4



In [19]:

# !python /content/road-damage-code/src/train.py

### YOLO26s Baseline Training Results

- **Training time:** 6.371 hours
- **Precision:** 65.2%
- **Recall:** 54.8%
- **mAP@50:** 59.7%
- **mAP@50–95:** 31.6%

The baseline training completed successfully. These results are from the validation set using `best.pt`. Final performance will be measured on the untouched test set.

### Baseline Model Analysis

- **Learning:** Training losses decreased steadily across 50 epochs.
- **Overfitting:** No major overfitting observed.
- **Performance on validation set**: began to plateau near the end.
- **Best class:** Alligator Crack (69.9% AP@50).
- **Weakest class:** Pothole (51.2% AP@50).
- **Class confusion:** Longitudinal and transverse cracks were sometimes confused.
- **Missed detections:** Some genuine road damages were missed, as seen in the Norwegian validation samples.

**Conclusion:** The baseline learned useful road-damage features but has room for improvement. Next, we investigate domain-wise performance and target specific weaknesses.

### Country-wise Validation Evaluation
Evaluate the trained YOLO26s model separately across all 7 RDD2022 domains.

In [20]:
#!python /content/road-damage-code/src/evaluate_by_country.py

### Country-wise Evaluation Findings

- **Best domain:** China MotorBike (91.7% mAP@50).
- **Weakest domain:** Norway (26.9% mAP@50, 25.9% recall).
- **India:** 42.0% mAP@50.
- Performance varies significantly across the 7 RDD2022 domains.
- Differences may reflect camera viewpoints, road conditions, and class distributions.

**Next:** Investigate Norway and India to identify targeted improvements.

## Experiment 2 — YOLO26m (Larger Model)

Evaluate whether using a larger capacity model improves overall road-damage
detection performance compared with our YOLO26s baseline.



### YOLO26m Full-Dataset Training Benchmark Test

A one-epoch benchmark test is performed on the complete dataset
to estimate GPU memory usage and training time on NVIDIA L4 GPU before full training.


This benchmark test is only for measuring training feasibility and cost.
It is not intended to produce the final YOLO26m model.


In [21]:
if False:

    import sys
    import time
    from ultralytics import YOLO

    sys.path.insert(0, "/content/road-damage-code")
    from src.train import prepare_data_yaml

    # Prepare the dataset configuration
    data_yaml = prepare_data_yaml()

    # Load the pretrained YOLO26m model
    model = YOLO("yolo26m.pt")

    # Run the 1-epoch training benchmark
    start_time = time.time()
    results = model.train(
        data=str(data_yaml),
        epochs=1,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        val=False,
        plots=False,
        close_mosaic=0,
        seed=42,
        project="/content/road-damage-code/runs",
        name="YOLO26m_benchmark"
    )

    elapsed_time = time.time() - start_time
    print(f"YOLO26m benchmark training completed in: {elapsed_time / 60:.2f} minutes")


### YOLO26m Benchmark Test Results

- **Total runtime:** 11.34 minutes
- **GPU memory:** 4.65 GB

The benchmark completed successfully on the NVIDIA L4 GPU.
YOLO26m fits comfortably in GPU memory, making full training feasible.


### YOLO26m Full Training

Train YOLO26m for 50 epochs using our existing training script.

#### Configuration

- **Model:** Pretrained YOLO26m
- **Dataset:** RDD2022 (30,704 train / 3,835 val)
- **Epochs:** 50
- **Image size:** 640 × 640
- **Optimizer:** Auto
- **Augmentation:** Ultralytics defaults
- **Batch size:** 8
- **GPU:** NVIDIA L4

**Hypothesis:** Increasing model capacity may improve overall
road-damage detection performance compared with our YOLO26s baseline.


In [22]:
#!python /content/road-damage-code/src/train_yolo26m.py


### YOLO26m Full Training Results


- **Precision:** 64.6%
- **Recall:** 55.7%
- **mAP@50:** 60.3%
- **mAP@50–95:** 31.9%

The YOLO26m training completed successfully.

Compared with the YOLO26s baseline, YOLO26m achieved only
a small improvement in overall validation mAP.

**Conclusion:** YOLO26m achieved a small validation improvement over
YOLO26s. Both checkpoints are retained; final model selection remains
open pending further experiments.




## Experiment 3 — Augmentation Tuning

YOLO26m showed only a small improvement over YOLO26s.
Instead of using a larger model, we will now test whether
changing data augmentation improves YOLO26s accuracy.

### New Training Configuration:

- **Model:** Pretrained YOLO26s
- **Dataset:** RDD2022 (30,704 train / 3,835 val)
- **Epochs:** 50
- **Image size:** 640 × 640
- **Batch size:** 8
- **GPU:** NVIDIA L4
- **Optimizer:** Auto
- **Mosaic:** 0.5  (Baseline model had 1 which is ultralytics default)
- **Scale:** 0.25  (Baseline model had 0.5 which is ultralytics default)


### Hypothesis

Reducing Mosaic and Scale augmentation may help YOLO26s learn small cracks more clearly and improve overall road-damage detection accuracy.


In [23]:
#!python /content/road-damage-code/src/train_yolo26s_aug_tuned.py


### Experiment 3 — Training Results

- **Training time:** 6.525 hours
- **Precision:** 63.5%
- **Recall:** 54.5%
- **mAP@50:** 58.7%
- **mAP@50–95:** 30.8%

### Findings

- Training completed successfully, but validation accuracy improved very little after epoch 40.
- Reducing Mosaic and Scale did not improve overall accuracy.

**Conclusion:** The original YOLO26s baseline performed slightly better.
We will keep the baseline as our preferred smaller model and
investigate dataset quality and small-object detection next.


## Experiment 4 — Reducing Negative Training Images

### Negative Image Audit

We examined the training dataset to understand how many images
had no annotations for our four road-damage classes(negative images).

### Key Findings

- **11,720 of 30,704** training images were negative (38.2%).
- Norway (64.8%), Czech (62.1%), and India (58.0%) had the highest negative-image percentages.
- Most sampled negative images appeared correctly labeled.

### Relationship With Model Accuracy

Comparing these findings with our earlier country-wise evaluation,
we noticed that countries with more negative images generally had
lower mAP scores.

For example:
- **China MotorBike:** 2.3% negatives → 91.7% mAP@50
- **Norway:** 64.8% negatives → 26.9% mAP@50

This shows a strong negative relationship, but does not prove that
the negative images caused the lower accuracy.

### Conclusion

We will test whether reducing negative training images improves
overall detection accuracy, while keeping all positive images
and the original validation and test sets unchanged.

In [24]:
# !python /content/road-damage-code/src/audit_negative_images.py

### Training yolo26s with Fewer Negative Images

**Hypothesis**: Reducing negative training images may help YOLO26s focus more on learning actual road damage, improving overall detection accuracy and recall.

### Changes From Baseline

- **Positive images:** 18,984 (unchanged)
- **Negative images:** 11,720 → 1,548
- **Total training images:** 30,704 → 20,532
- **Negative percentage:** 38.2% → 7.54%
- Negative images are limited to approximately 10% per domain where possible.
- Validation and test sets remain unchanged.

### Training Configuration

- **Model:** Pretrained YOLO26s
- **Epochs:** 50
- **Image size:** 640 × 640
- **Batch size:** 8
- **GPU:** NVIDIA L4
- **Optimizer:** Auto
- **Augmentation:** Default settings
- **Seed:** 42

In [25]:
#!python /content/road-damage-code/src/train_yolo26s_neg10.py


### Experiment 4 — Training Results

- **Training time:** 4.33 hours
- **Precision:** 61.7%
- **Recall:** 56.6%
- **mAP@50:** 58.9%
- **mAP@50–95:** 31.3%

### Findings

- Reducing negative images slightly improved recall compared with the baseline.
- Overall mAP did not improve.
- The performance on the validation plateaued near the end of training.

**Conclusion:** Reducing negative images from 38.2% to 7.54% did
not improve overall mAP. The original YOLO26s remains our preferred
smaller model.

This experiment also used fewer training batches per epoch, so
the effect of negative-image balance cannot be fully isolated.


## Experiment 5 — Fine-Tuning With Norway Crops

### Creating Norway Training Crops

Our earlier country-wise evaluation identified Norway as a weak domain.
Its high-resolution images contain small defects that become difficult
to distinguish when resized for training.

**Hypothesis:** Adding closer views of Norwegian road damage may help
the model learn these smaller defects more effectively.

### Crop Preparation

- Extract a **1,824 × 1,824** crop from the lower-left region of eligible Norwegian training images.
- Recalculate bounding boxes relative to each crop.

### Dataset Protection

Cropped images and labels are saved separately. The original training
images, validation set, and test set remain unchanged.

These crops provide additional views of existing training examples;
they are not new independent road scenes.

In [26]:
#!python /content/road-damage-code/src/create_norway_crops.py

### Combining Original Images With Norway Crops

Create a training manifest: a text file listing the image paths that
YOLO will use for this experiment.

The manifest combines the complete original training set with the
additional Norway crops.



**Conclusion:** The model will learn from both the original images and
closer views of selected Norwegian defects, while evaluation continues
on the original full-image validation set.

In [27]:
#!python /content/road-damage-code/src/create_crop_training_manifest.py

### Fine-Tuning YOLO26s With Norway Crops

**Hypothesis:** Further training on the original images plus Norway
crops may improve small-defect detection without losing performance
on the other countries.

### Changes From Baseline

- Start from our trained YOLO26s baseline `best.pt`.
- Increase training entries from **30,704 to 32,845**.
- Switch to **AdamW** with an initial learning rate of **0.0005**.
- Disable mosaic throughout this fine-tuning run.
- Keep the original validation and test sets unchanged.

### Training Configuration

- **Model:** Trained YOLO26s baseline
- **Scheduled epochs:** 25
- **Image size:** 640 × 640
- **Batch size:** 8
- **Optimizer:** AdamW
- **Initial learning rate:** 0.0005
- **Mosaic:** Disabled
- **Other augmentations:** Default settings
- **Seed:** 42

Checkpoints and training logs are saved separately under
`YOLO26s_norway_crops_ft25`, preserving the original baseline weights.

In [28]:
#!python /content/road-damage-code/src/train_yolo26s_norway_crops.py

### Experiment 5 — Training Results

The metrics below are from **epoch 16**, which achieved the highest
recorded validation mAP@50–95.

- **Precision:** 64.1%
- **Recall:** 54.0%
- **mAP@50:** 58.2%
- **mAP@50–95:** 30.14%
- **Original YOLO26s baseline mAP@50–95:** 31.55%
- **Change from baseline:** −1.41 percentage points

### Findings

- Validation performance dropped sharply at the beginning of fine-tuning.
- Performance gradually recovered but remained below the original baseline.
- The best recorded mAP@50–95 occurred at epoch 16; later recorded epochs did not improve it.
- Overall metrics alone do not establish whether Norway specifically improved.

**Conclusion:** This fine-tuning configuration did not improve overall
validation mAP@50–95. The original YOLO26s remains the preferred
smaller model.


## Experiment 6 — Fine-tuning YOLO26m at 960 Pixels

**Hypothesis:** A larger input may preserve details that are harder to see at 640 pixels, especially thin cracks and small potholes.

### Changes From Baseline

- **Starting weights:** `best.pt` from the original 50-epoch YOLO26m run.
- **Input size:** 640 → 960 pixels. The full training split and original validation set stayed the same.
- **Training configuration:** Batch size 4, MuSGD optimizer, learning rate 0.0003, and mosaic disabled.
- **Pilot:** Used a 20-epoch schedule but paused after 3 epochs to review the results before spending more GPU credits.

### Evaluation Plan

Evaluate the original and fine-tuned checkpoints at both 640 and 960. Comparing them at the same input size shows what fine-tuning changed. Comparing both against the original at 640 shows whether the new setup is worth keeping.

In [29]:
if False:


      from pathlib import Path
      import sys, json, hashlib, importlib.metadata
      import torch, ultralytics, yaml
      from ultralytics import YOLO

      project = Path("/content/road-damage-code")
      sys.path.insert(0, str(project))
      from src.train import prepare_data_yaml

      assert ultralytics.__version__ == importlib.metadata.version("ultralytics") == "8.4.204", (
          "Use %pip install -q ultralytics==8.4.204 pyyaml in your existing installation cell, "
          "then restart the Python session and rerun your existing setup cells."
      )
      assert torch.cuda.is_available(), "Select a GPU runtime before running."
      drive_runs = Path("/content/drive/MyDrive/road-damage-detection/training_runs")
      assert drive_runs.is_dir(), "Run your existing Drive mount cell first."
      weights = drive_runs / "YOLO26m_full50/weights/best.pt"
      assert weights.is_file(), weights
      data_yaml = Path(prepare_data_yaml())
      data = yaml.safe_load(data_yaml.read_text())
      assert data["train"] == "images/train", "Use the original full training set, not the crop/subset manifest."
      run_name = "YOLO26m_full_data_960_lowLR20"
      run_dir = drive_runs / run_name
      evidence = drive_runs / (run_name + "_comparison")
      evidence.mkdir(exist_ok=True)

      def digest(path):
          h = hashlib.sha256()
          with Path(path).open("rb") as f:
              for b in iter(lambda: f.read(1024 * 1024), b""): h.update(b)
          return h.hexdigest()

      eval_args = dict(data=str(data_yaml), split="val", batch=8, device=0,
          conf=0.001, iou=0.7, max_det=300, nms=True, augment=False,
          rect=True, half=False, plots=False, project=str(evidence))

      def evaluate(path, size, name):
          m = YOLO(str(path)).val(**eval_args, imgsz=size, name=name)
          return dict(precision=float(m.box.mp), recall=float(m.box.mr),
              map50=float(m.box.map50), map5095=float(m.box.map),
              per_class_ap5095={str(m.names[i]): float(v) for i,v in enumerate(m.box.maps)},
              speed_ms=m.speed)

      # These runs take minutes rather than another training session.
      # They establish the SAME environment and SAME resolution for the comparison.
      identity = dict(weights_sha256=digest(weights), ultralytics=ultralytics.__version__,
          torch=torch.__version__, data=data)
      before = {"identity":identity, "640":evaluate(weights,640,"before_640"),
                "960":evaluate(weights,960,"before_960")}
      (evidence / "before.json").write_text(json.dumps(before,indent=2))
      print("Original medium, 640:", round(before["640"]["map5095"]*100,3))
      print("Original medium, 960:", round(before["960"]["map5095"]*100,3))
      assert abs(before["640"]["map5095"]-.3194) < .005, (
          "Medium baseline differs by >0.5 percentage points. Resolve comparison before training."
      )


In [30]:
if False:


      import shutil

      # Leave this at 3 for the first run. After Cell 3 passes the spending gate,
      # change it to 8 for the next stage, then at most 20 after another review.
      STOP_AFTER_EPOCH = 3
      assert STOP_AFTER_EPOCH in (3, 8, 20)

      class PauseWithCheckpoint:
          def __init__(self, end_epoch):
              self.end_epoch = end_epoch

          def __call__(self, trainer):
              # on_model_save occurs before final_eval strips optimizer state.
              if trainer.epoch + 1 >= self.end_epoch or trainer.stop:
                  destination = Path(trainer.last).parent / "resume_before_strip.pt"
                  temporary = destination.with_suffix(".tmp")
                  shutil.copy2(trainer.last, temporary)
                  temporary.replace(destination)
                  (destination.parent / "pause_reason.json").write_text(json.dumps({
                      "completed_epochs": trainer.epoch + 1,
                      "stage_limit": self.end_epoch,
                      "early_stopping_before_stage": bool(trainer.stop and trainer.epoch + 1 < self.end_epoch),
                  }, indent=2))
                  trainer.stop = True
                  print(f"Paused after epoch {trainer.epoch + 1}; resumable checkpoint: {destination}")

      resume_path = None
      completed = 0
      if run_dir.exists():
          reason_file = run_dir / "weights/pause_reason.json"
          if reason_file.is_file():
              reason = json.loads(reason_file.read_text())
              assert not reason["early_stopping_before_stage"], (
                  "Natural early stopping ended this run. Review it instead of resuming automatically."
              )
          candidates = []
          for p in (run_dir / "weights/last.pt", run_dir / "weights/resume_before_strip.pt"):
              if not p.is_file():
                  continue
              # Only load checkpoints created by YOUR run; these are trusted local files.
              ckpt = torch.load(p, map_location="cpu", weights_only=False)
              if ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) >= 0:
                  assert ckpt["train_args"]["epochs"] == 20
                  candidates.append((int(ckpt["epoch"]) + 1, p))
              del ckpt
          assert candidates, "No resumable checkpoint found. Do not start another run automatically."
          completed, resume_path = max(candidates, key=lambda pair: pair[0])
          assert completed < STOP_AFTER_EPOCH, (
              f"Already reached epoch {completed}. Run Cell 3 and review before extending the stage."
          )
      else:
          assert STOP_AFTER_EPOCH == 3, "Start with the three-epoch spending limit."

      if STOP_AFTER_EPOCH > 3:
          comparison = json.loads((evidence / "comparison.json").read_text())
          assert comparison["candidate_sha256"] == digest(run_dir / "weights/best.pt"), (
              "Run Cell 3 again: the current best checkpoint has not been compared yet."
          )
          assert comparison["spending_gate"]["pass"], (
              "The comparison did not pass the spending gate. Pause and inspect the results."
          )

      model = YOLO(str(resume_path or weights))
      model.add_callback("on_model_save", PauseWithCheckpoint(STOP_AFTER_EPOCH))
      if resume_path is not None:
          model.train(resume=True, data=str(data_yaml), device=0)
      else:
          model.train(
          data=str(data_yaml), epochs=20, imgsz=960, batch=4, nbs=64,
          device=0, workers=2, seed=42, deterministic=True,
          optimizer="MuSGD", lr0=0.0003, lrf=0.1, cos_lr=True,
          momentum=0.9, weight_decay=0.0005,
          warmup_epochs=1.0, warmup_bias_lr=0.0, warmup_momentum=0.8,
          mosaic=0.0, close_mosaic=0, scale=0.1, translate=0.1,
          mixup=0.0, copy_paste=0.0, cutmix=0.0,
          hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
          fliplr=0.5, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
          multi_scale=0.0, freeze=None, rect=False,
          nms=True, conf=0.001, iou=0.7, max_det=300,
          amp=True, cache=False, val=True, patience=8,
          save=True, save_period=5, plots=True,
          resume=False, project=str(drive_runs), name=run_name, exist_ok=False,
          )


In [31]:
if False:


      candidate = run_dir / "weights/best.pt"
      assert candidate.is_file(), candidate
      before = json.loads((evidence / "before.json").read_text())
      assert digest(weights) == before["identity"]["weights_sha256"], "Original weights changed."
      after = {"640": evaluate(candidate,640,"after_640"),
              "960": evaluate(candidate,960,"after_960")}
      reference = max(before["640"]["map5095"], before["960"]["map5095"])
      gain = after["960"]["map5095"] - reference
      class_deltas = {
          k: after["960"]["per_class_ap5095"][k] - v
          for k, v in before["640"]["per_class_ap5095"].items()
      }
      # Conservative spending rules, NOT statistical confidence intervals or promises.
      gate = dict(gain_points=100*gain, minimum_gain_points=0.3,
                  class_deltas_points={k:100*v for k,v in class_deltas.items()},
                  maximum_allowed_class_drop_points=2.0)
      gate["pass"] = bool(gain >= 0.003 and min(class_deltas.values()) >= -0.02)
      report = {"before":before,"after":after,"candidate_sha256":digest(candidate),
                "spending_gate":gate}
      (evidence / "comparison.json").write_text(json.dumps(report,indent=2))
      for size in ("640","960"):
          a,b = before[size]["map5095"],after[size]["map5095"]
          print(f"{size}: original {a*100:.3f}% -> trained {b*100:.3f}% ({(b-a)*100:+.3f} points)")
      print("New 960 vs original 640:",
            round(100*(after["960"]["map5095"]-before["640"]["map5095"]),3),"points")
      print("Full metrics and class AP:",evidence / "comparison.json")
      print("Class AP changes vs original 640:", gate["class_deltas_points"])
      print("Spending gate:", "PASS — review runtime before extending" if gate["pass"]
            else "PAUSE — do not spend more credits automatically")
      print("This gate tests whether to invest more, not whether the model could ever improve.")


### Experiment 6 — Training Results

| Input Size | Original YOLO26m | Fine-tuned YOLO26m | Change |
|---|---:|---:|---:|
| 640 | 32.035% | 30.713% | −1.322 pts |
| 960 | 28.996% | 29.923% | +0.927 pts |

*All scores are validation mAP@50–95.*

### Findings

- Increasing the input size alone lowered the original model’s score from **32.035% to 28.996%**.
- Fine-tuning recovered some of that loss at 960, but the result still finished **2.112 percentage points below** the original at 640.
- All four classes scored lower when comparing the fine-tuned model at 960 with the original at 640. Alligator cracks had the largest drop (**−4.36 points**).

**Conclusion:** The 960-pixel pilot improved on its own starting score but did not beat our existing 640-pixel setup. We kept the original checkpoint and did not extend the run.

## Experiment 7 — Frozen Backbone and Class Weighting

**Hypothesis:** Freezing the backbone may preserve useful features from the original YOLO26m, while giving rarer classes more weight may help the model detect damage such as potholes.

### Changes From Baseline

- **Starting weights:** Original YOLO26m `best.pt`.
- **Backbone:** Froze the first 11 blocks (`freeze=11`) and fine-tuned the remaining layers.
- **Class weighting:** Set `cls_pw=0.5`. Weights came from the training split, with potholes receiving the highest weight.
- **Training configuration:** 640 pixels, batch size 8, MuSGD, learning rate 0.0001, seed 42, and up to 8 epochs.

### Evaluation Plan

Compare the best trial checkpoint with the original model on the same validation set at 640 pixels. The trial needed at least **+0.5 percentage points in mAP@50–95**, with no class dropping by more than **1 point**.

In [32]:
if False:


      from pathlib import Path
      import hashlib, importlib.metadata, json, shutil, sys, time
      import torch, ultralytics, yaml
      from ultralytics import YOLO

      assert ultralytics.__version__ == importlib.metadata.version('ultralytics') == '8.4.204'
      project = Path('/content/road-damage-code')
      sys.path.insert(0, str(project))
      from src.train import prepare_data_yaml

      drive_runs = Path('/content/drive/MyDrive/road-damage-detection/training_runs')
      original_weights = drive_runs / 'YOLO26m_full50/weights/best.pt'
      reference_file = drive_runs / 'YOLO26m_full_data_960_lowLR20_comparison/before.json'
      assert original_weights.is_file() and reference_file.is_file(), 'Restore the existing Drive mount/setup.'
      data_yaml = Path(prepare_data_yaml())
      data_config = yaml.safe_load(data_yaml.read_text())
      reference = json.loads(reference_file.read_text())

      def sha256(path):
          h = hashlib.sha256()
          with Path(path).open('rb') as f:
              for block in iter(lambda: f.read(1024 * 1024), b''):
                  h.update(block)
          return h.hexdigest()

      assert sha256(original_weights) == reference['identity']['weights_sha256'], 'Original checkpoint changed.'
      assert json.loads(json.dumps(data_config)) == reference["identity"]["data"], (
          "Dataset configuration differs from the saved comparison."
      )
      assert data_config['train'] == 'images/train', 'Use the original complete training split.'
      baseline_score = reference['640']['map5095']
      assert abs(baseline_score - 0.32035) < 0.0001, 'This is not the expected saved medium baseline.'

      # Candidate first. The optional CONTROL is a separate run FROM THE SAME ORIGINAL
      # CHECKPOINT, with cls_pw=0.0. Do not run it automatically or initialize it from the candidate.
      ARM = 'balanced'
      assert ARM in ('balanced', 'control')
      CLASS_POWER = 0.5 if ARM == 'balanced' else 0.0
      run_name = f'YOLO26m_640_freeze11_{ARM}_8ep'
      run_dir = drive_runs / run_name
      report_dir = drive_runs / (run_name + '_comparison')
      report_dir.mkdir(exist_ok=True)

      print(f'Original medium reference: {baseline_score*100:.3f}% mAP50-95')
      print('Eight-epoch trial at 640; backbone frozen; original full training data.')
      print(f'Arm: {ARM}; cls_pw={CLASS_POWER}. Cell 2 starts paid GPU training.')



In [33]:
if False:


      assert torch.cuda.is_available(), 'A GPU is required for this training cell.'
      RESUME = False  # Set True ONLY for an interruption of this exact eight-epoch trial.
      MAX_TRAIN_MINUTES = 240.0  # Epoch-boundary budget; setup/final evaluation are additional.
      resume_file = run_dir / 'weights/resume_before_strip.pt'
      progress_file = report_dir / 'progress.json'
      previous = json.loads(progress_file.read_text()) if progress_file.exists() else {}

      if run_dir.exists():
          assert RESUME, 'This run exists. Do not overwrite or restart it.'
          assert previous.get('stop_reason') is None, 'A deliberate stop requires review, not automatic resumption.'
          candidates = []
          for path in (run_dir / 'weights/last.pt', resume_file):
              if path.is_file():
                  ckpt = torch.load(path, map_location='cpu', weights_only=False)  # Your own trusted checkpoints.
                  if ckpt.get('optimizer') is not None and 0 <= ckpt.get('epoch', -1) < 7:
                      args = ckpt['train_args']
                      assert args['epochs'] == 8 and args['imgsz'] == 640
                      assert args['freeze'] == 11 and args['cls_pw'] == CLASS_POWER
                      candidates.append((ckpt['epoch'], path))
                  del ckpt
          assert candidates, 'No unfinished optimizer-bearing checkpoint found.'
          training_weights = max(candidates, key=lambda x: x[0])[1]
      else:
          assert not RESUME, 'There is no trial to resume.'
          training_weights = original_weights

      class TrialMonitor:
          def __init__(self, previous, budget_minutes, baseline, progress_file):
              self.previous = previous
              self.budget = budget_minutes
              self.baseline = baseline
              self.progress_file = progress_file
              self.checked_epoch = None
              self.low_scores = int(previous.get('consecutive_large_drops', 0))

          def start(self, trainer):
              self.started = time.monotonic()
              self.prior_minutes = float(self.previous.get('training_minutes', 0.0))
              self.last_minutes = self.prior_minutes
              assert len(trainer.train_loader.dataset) == 30704
              assert len(trainer.test_loader.dataset) == 3835
              assert trainer.args.batch == 8, 'Batch changed; review before training.'
              if CLASS_POWER:
                  weights = trainer.model.class_weights.detach().cpu().tolist()
                  print('Train-only class weights:', dict(zip(trainer.model.names.values(), weights)))
              assert self.prior_minutes < self.budget, 'Training budget already used.'

          def check_frozen(self, trainer):
              if self.checked_epoch == trainer.epoch:
                  return
              # This hook runs AFTER the trainer switches to train mode each epoch.
              for block in trainer.model.model[:11]:
                  assert not any(p.requires_grad for p in block.parameters()), 'Backbone is not frozen.'
                  for module in block.modules():
                      if isinstance(module, torch.nn.BatchNorm2d):
                          assert not module.training, 'Frozen backbone BN statistics are updating.'
              self.checked_epoch = trainer.epoch

          def saved(self, trainer):
              elapsed = self.prior_minutes + (time.monotonic() - self.started) / 60
              epoch_minutes = elapsed - self.last_minutes
              self.last_minutes = elapsed
              score = float(trainer.metrics['metrics/mAP50-95(B)'])
              # A loss-containment rule, not a statistical test or proof of futility.
              self.low_scores = self.low_scores + 1 if score < self.baseline - 0.02 else 0
              completed = trainer.epoch + 1
              reason = None
              if completed >= 8:
                  reason = 'eight_epoch_trial_complete'
              elif self.low_scores >= 2:
                  reason = 'two_consecutive_epochs_more_than_2_points_below_reference'
              elif elapsed + epoch_minutes > self.budget:
                  reason = 'next_epoch_would_exceed_estimated_training_budget'
              elif trainer.stop:
                  reason = 'trainer_stopped'
              if reason:
                  destination = Path(trainer.last).parent / 'resume_before_strip.pt'
                  temporary = destination.with_suffix('.tmp')
                  shutil.copy2(trainer.last, temporary)
                  temporary.replace(destination)
                  trainer.stop = True
                  print('Trial stopped:', reason)
              self.progress_file.write_text(json.dumps(dict(
                  completed_epochs=completed, training_minutes=elapsed,
                  last_epoch_minutes=epoch_minutes, monitored_map5095=score,
                  consecutive_large_drops=self.low_scores, stop_reason=reason,
              ), indent=2))

      model = YOLO(str(training_weights))
      monitor = TrialMonitor(previous, MAX_TRAIN_MINUTES, baseline_score, progress_file)
      model.add_callback('on_train_start', monitor.start)
      model.add_callback('on_train_batch_start', monitor.check_frozen)
      model.add_callback('on_model_save', monitor.saved)

      if RESUME:
          model.train(resume=True, data=str(data_yaml), device=0)
      else:
          # Keep the original baseline's ordinary augmentation and optimizer settings
          # where available. The baseline's final phase already had mosaic closed.
          original_args = model.ckpt['train_args']
          preserved_defaults = dict(
              scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
              fliplr=0.5, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
              mixup=0.0, copy_paste=0.0, cutmix=0.0,
              momentum=0.937, weight_decay=0.0005,
          )
          preserved = {k: original_args.get(k, v) for k, v in preserved_defaults.items()}
          model.train(
              data=str(data_yaml), epochs=8, imgsz=640, batch=8, nbs=64,
              freeze=11, cls_pw=CLASS_POWER,
              optimizer='MuSGD', lr0=0.0001, lrf=0.1, cos_lr=True,
              warmup_epochs=1.0, warmup_bias_lr=0.0, warmup_momentum=0.8,
              mosaic=0.0, close_mosaic=0, multi_scale=0.0, rect=False,
              box=7.5, cls=0.5, dfl=1.5,
              nms=True, conf=0.001, iou=0.7, max_det=300,
              device=0, workers=2, seed=42, deterministic=True,
              amp=True, val=True, patience=8, cache=False, fraction=1.0,
              save=True, save_period=-1, plots=True,
              project=str(drive_runs), name=run_name, exist_ok=False,
              **preserved,
          )



In [34]:
if False:

      candidate = run_dir / 'weights/best.pt'
      assert candidate.is_file(), 'Run Cell 2 first.'
      assert sha256(original_weights) == reference['identity']['weights_sha256']
      metrics = YOLO(str(candidate)).val(
          data=str(data_yaml), split='val', imgsz=640, batch=8, device=0,
          conf=0.001, iou=0.7, max_det=300, nms=True, augment=False,
          rect=True, half=False, plots=False, project=str(report_dir), name='matched_validation',
      )
      after = dict(
          precision=float(metrics.box.mp), recall=float(metrics.box.mr),
          map50=float(metrics.box.map50), map5095=float(metrics.box.map),
          per_class_ap5095={str(metrics.names[i]): float(v) for i, v in enumerate(metrics.box.maps)},
          speed_ms=metrics.speed,
      )
      gain = 100 * (after['map5095'] - baseline_score)
      deltas = {k: 100 * (after['per_class_ap5095'][k] - v)
                for k, v in reference['640']['per_class_ap5095'].items()}
      promising = gain >= 0.5 and min(deltas.values()) >= -1.0
      report = dict(
          arm=ARM, before=reference, after=after, gain_points=gain,
          per_class_change_points=deltas, candidate_sha256=sha256(candidate),
          promising_for_further_review=promising,
          decision_rule='At least +0.5 mAP50-95 points and no class drop greater than 1 point; not significance.',
          remaining='Country-wise comparison and untouched-test evaluation before a final delivery claim.',
      )
      (report_dir / 'comparison.json').write_text(json.dumps(report, indent=2))
      print(f'Original medium: {baseline_score*100:.3f}%')
      print(f'New {ARM} model: {after["map5095"]*100:.3f}% ({gain:+.3f} points)')
      print('Per-class AP50-95 changes:', deltas)
      print('PROMISING: review before any further training.' if promising
            else 'STOP: keep the original model; no automatic extension or new run.')
      print('Saved:', report_dir / 'comparison.json')


### Experiment 7 — Training Results

| Validation Metric | Original YOLO26m | Fine-tuned YOLO26m |
|---|---:|---:|
| Precision | 64.53% | 64.01% |
| Recall | 55.79% | 55.67% |
| mAP@50 | 60.35% | 60.29% |
| mAP@50–95 | 32.035% | 31.662% |

### Findings

- Overall mAP@50–95 fell by **0.373 percentage points**, missing the improvement target.
- All four classes scored slightly lower. Transverse cracks had the largest drop (**−0.58 points**).
- Pothole AP@50–95 fell from **23.25% to 22.87%**, despite potholes receiving the highest class weight.

**Conclusion:** The combination of freezing and class weighting did not improve this checkpoint. Since both changes were tested together, we cannot tell which contributed to the drop. We kept the original YOLO26m at 640 pixels.

## Experiment 8 — Combining YOLO26s and YOLO26m Predictions

### Experiment Setup

**Hypothesis:** The small and medium models may detect different road defects. Combining their predictions could improve detection compared with either model alone.

We use the original `best.pt` checkpoint from each model and keep the input size at **640 pixels**. This experiment uses the existing validation split and requires no additional training.

**Before running:** Complete the Colab setup above and make sure both original `best.pt` checkpoints and Experiment 6’s `before.json` are available in Google Drive. The earlier experiment cells are disabled, so **Run all** will not create these files.

In [35]:
if False:


        %pip install -q ensemble-boxes==1.0.9
        from pathlib import Path
        import gzip, hashlib, json
        from importlib.metadata import version
        import numpy as np
        import pandas as pd
        import torch
        import ultralytics
        from ultralytics import YOLO
        from ultralytics.models.yolo.detect import DetectionValidator
        from ultralytics.utils.metrics import DetMetrics
        from ensemble_boxes import weighted_boxes_fusion
        from IPython.display import display

        import sys, yaml
        project = Path('/content/road-damage-code')
        sys.path.insert(0,str(project))
        from src.train import prepare_data_yaml
        assert ultralytics.__version__ == version("ultralytics") == "8.4.204", "Use the existing pinned setup."
        runs = Path('/content/drive/MyDrive/road-damage-detection/training_runs')
        weights = {k:runs/d/'weights/best.pt' for k,d in [('YOLO26s','YOLO26s_baseline_model'),('YOLO26m','YOLO26m_full50')]}
        for p in weights.values(): assert p.is_file(), p
        original_yaml = Path(prepare_data_yaml())
        ref=json.loads((runs/'YOLO26m_full_data_960_lowLR20_comparison/before.json').read_text())
        assert json.loads(json.dumps(yaml.safe_load(original_yaml.read_text()))) == ref['identity']['data']
        assert hashlib.sha256(weights['YOLO26m'].read_bytes()).hexdigest() == ref['identity']['weights_sha256']
        wbf_dir = runs/'final_s_m_WBF_640'
        wbf_dir.mkdir(exist_ok=True)
        eval_settings=dict(data=str(original_yaml), imgsz=640,batch=8,device=0 if torch.cuda.is_available() else 'cpu',
                          conf=0.001,iou=0.7,max_det=300,nms=True,rect=True,augment=False,plots=False)
        settings = dict(eval_settings, split="val", half=False, verbose=False,
                        save_json=False, save_txt=False, exist_ok=True,
                        project=str(wbf_dir))

        # A quick file signature prevents reusing predictions after data changes.
        data_root = project / "data/processed/main_split"
        signature = hashlib.sha256(Path(original_yaml).read_bytes())
        for folder in ("images/val", "labels/val"):
            for path in sorted((data_root / folder).rglob("*")):
                if path.is_file():
                    signature.update(str(path.relative_to(data_root)).encode())
                    signature.update(str((path.stat().st_size, path.stat().st_mtime_ns)).encode())
                    if path.suffix == ".txt":
                        signature.update(path.read_bytes())
        environment = {"loaded": ultralytics.__version__,
                      "installed": version("ultralytics"), "torch": torch.__version__}
        print("Using current session:", environment)
        print("Results folder:", wbf_dir)

### Collecting Predictions From Both Models

Run YOLO26s and YOLO26m separately on the same **3,835 validation images** and save their predicted boxes, damage classes, and confidence scores.

Saving both sets of predictions lets us compare three approaches on exactly the same images: **YOLO26s alone**, **YOLO26m alone**, and **both models combined**. The saved predictions can be reused without running either model again.

In [36]:
if False:


        def collect_predictions(tag, checkpoint):
            identity = dict(environment=environment, settings=settings,
                            data=signature.hexdigest(),
                            weights=hashlib.sha256(checkpoint.read_bytes()).hexdigest())
            key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
            cache = wbf_dir / f"{tag}_{key}.json.gz"
            if cache.is_file():
                print("Reusing saved predictions:", tag)
                with gzip.open(cache, "rt") as file:
                    return json.load(file)

            records = {}

            class CaptureValidator(DetectionValidator):
                def update_metrics(self, predictions, batch):
                    for i, pred in enumerate(predictions):
                        target = self._prepare_batch(i, batch)
                        image = str(target["im_file"])
                        if image in records:
                            raise RuntimeError(f"Repeated validation image: {image}")
                        records[image] = {
                            "shape": list(batch["img"].shape[2:]),
                            "boxes": pred["bboxes"].detach().cpu().tolist(),
                            "scores": pred["conf"].detach().cpu().tolist(),
                            "classes": pred["cls"].detach().cpu().tolist(),
                            "gt_boxes": target["bboxes"].detach().cpu().tolist(),
                            "gt_classes": target["cls"].detach().cpu().tolist(),
                        }
                    super().update_metrics(predictions, batch)

            model = YOLO(str(checkpoint))
            metrics = model.val(validator=CaptureValidator, name=tag, **settings)
            if len(records) != 3835 or sum(len(r["gt_classes"]) for r in records.values()) != 5520:
                raise RuntimeError("Validation image/annotation counts changed. Stop here.")
            result = dict(identity=identity, names=model.names,
                          native=dict(metrics.results_dict), records=records)
            temporary = cache.with_suffix(".tmp")
            with gzip.open(temporary, "wt") as file:
                json.dump(result, file, allow_nan=False)
            temporary.replace(cache)
            return result

        cached = {tag: collect_predictions(tag, path) for tag, path in weights.items()}
        print("Both models' predictions are saved. Run Cell 3.")

### Combining Predictions With Weighted Boxes Fusion

Weighted Boxes Fusion (WBF) groups overlapping boxes when both models predict the same damage class. It uses their confidence scores to calculate one box, giving more influence to the more confident prediction.

Before combining the boxes, we check that our scoring code reproduces each model’s individual validation results. We then compare overall performance, results for each damage class, and results across all seven RDD2022 domains.

In [37]:
if False:

      names = {int(k): v for k, v in cached["YOLO26s"]["names"].items()}
      other_names = {int(k): v for k, v in cached["YOLO26m"]["names"].items()}
      if names != other_names:
          raise RuntimeError("The models have different class mappings.")
      torch.set_num_threads(min(4, torch.get_num_threads()))
      matcher = DetectionValidator(args={"task": "detect", "plots": False})

      def score(records):
          metrics = DetMetrics(names=names)
          for image, row in records.items():
              pred = {"bboxes": torch.tensor(row["boxes"], dtype=torch.float32).reshape(-1, 4),
                      "cls": torch.tensor(row["classes"], dtype=torch.float32)}
              target = {"bboxes": torch.tensor(row["gt_boxes"], dtype=torch.float32).reshape(-1, 4),
                        "cls": torch.tensor(row["gt_classes"], dtype=torch.float32)}
              gt = np.asarray(row["gt_classes"])
              metrics.update_stats({**matcher._process_batch(pred, target),
                                    "conf": np.asarray(row["scores"],dtype=np.float32),
                                    "pred_cls": np.asarray(row["classes"]),
                                    "target_cls": gt, "target_img": np.unique(gt),
                                    "im_name": image})
          metrics.process(plot=False)
          return metrics

      datasets = {tag: item["records"] for tag, item in cached.items()}
      measured = {tag: score(records) for tag, records in datasets.items()}
      for tag in measured:
          for metric in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
              if abs(measured[tag].results_dict[metric] - cached[tag]["native"][metric]) > 0.0001:
                  raise RuntimeError(f"{tag}: cached scoring differs from YOLO. Stop here.")
      print("Scoring check passed for both models.")

      def fuse(first, second):
          if first.keys() != second.keys():
              raise RuntimeError("The validation image sets differ.")
          combined = {}
          for image, a in first.items():
              b = second[image]
              if any(a[k] != b[k] for k in ("shape", "gt_boxes", "gt_classes")):
                  raise RuntimeError(f"Image geometry or labels differ: {image}")
              h, w = a["shape"]
              scale = np.array([w, h, w, h])
              boxes_list, scores_list, labels_list = [], [], []
              for row in (a, b):
                  boxes = np.clip(np.asarray(row["boxes"]).reshape(-1, 4) / scale, 0, 1)
                  keep = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
                  boxes_list.append(boxes[keep].tolist())
                  scores_list.append(np.asarray(row["scores"],dtype=np.float32)[keep].tolist())
                  labels_list.append(np.asarray(row["classes"])[keep].tolist())
              boxes, confidence, labels = weighted_boxes_fusion(
                  boxes_list, scores_list, labels_list, weights=[1, 1],
                  iou_thr=0.55, skip_box_thr=0.001, conf_type="avg")
              order = np.argsort(-confidence, kind="stable")[:300]
              combined[image] = dict(a, boxes=(boxes[order] * scale).tolist(),
                                    scores=confidence[order].tolist(), classes=labels[order].tolist())
          return combined

      print("Combining predictions on CPU...")
      datasets["WBF"] = fuse(datasets["YOLO26s"], datasets["YOLO26m"])
      measured["WBF"] = score(datasets["WBF"])
      overall = pd.DataFrame({tag: {"Precision": m.box.mp * 100, "Recall": m.box.mr * 100,
                                  "mAP50": m.box.map50 * 100, "mAP50-95": m.box.map * 100}
                              for tag, m in measured.items()}).T
      per_class = pd.DataFrame({tag: {names[i]: float(v)*100 for i,v in enumerate(m.box.maps)} for tag,m in measured.items()})
      countries = ("China_Drone", "China_MotorBike", "Czech", "India", "Japan", "Norway", "United_States")
      groups = {}
      for image in datasets["YOLO26s"]:
          country = next((c for c in countries if Path(image).name.startswith(c + "_")), "Unknown")
          groups.setdefault(country, []).append(image)
      by_country = pd.DataFrame({tag: {country: score({p: records[p] for p in images}).box.map * 100
                                    for country, images in groups.items()}
                                for tag, records in datasets.items()})

      print("Overall — all values are percentages:")
      display(overall.round(2))
      print("Per-class mAP50-95:")
      display(per_class.round(2))
      print("Country mAP50-95:")
      display(by_country.round(2))
      strongest = overall.loc[["YOLO26s", "YOLO26m"], "mAP50-95"].idxmax()
      gain = overall.loc["WBF", "mAP50-95"] - overall.loc[strongest, "mAP50-95"]
      print(f"WBF change versus {strongest}: {gain:+.2f} percentage points")
      reproduced = abs(overall.loc["YOLO26m","mAP50-95"] - 32.035) <= 0.1 and abs(overall.loc["YOLO26s","mAP50-95"] - 31.55) <= 0.1
      passed = bool(reproduced and gain >= 0.5 and (per_class["WBF"]-per_class[strongest]).min() >= -1.0)
      print("CHECK BASELINE REPRODUCTION" if not reproduced else ("PROMISING: review country results and video speed" if passed else "WBF did not pass the validation gate. See the final model selection below."))
      report = dict(passed_validation_gate=passed, gain_points=float(gain), benchmark=strongest, environment=environment, fusion=dict(weights=[1, 1], iou=0.55, conf=0.001),
                    overall=overall.to_dict(), per_class=per_class.to_dict(),
                    by_country=by_country.to_dict(), cache_keys={k: v["identity"] for k, v in cached.items()})
      (wbf_dir / "comparison.json").write_text(json.dumps(report, indent=2, allow_nan=False))
      with gzip.open(wbf_dir / "wbf_predictions.json.gz", "wt") as file:
          json.dump(datasets["WBF"], file, allow_nan=False)
      print("Saved:", wbf_dir / "comparison.json")

### Experiment 8 — Results and Conclusion

| Approach | Precision | Recall | mAP@50 | mAP@50–95 |
|---|---:|---:|---:|---:|
| YOLO26s | 65.60% | 54.77% | 59.78% | 31.55% |
| YOLO26m | 64.53% | 55.78% | 60.34% | 32.04% |
| YOLO26s + YOLO26m (WBF) | 64.86% | 55.69% | 60.02% | 31.24% |

### Findings

- Fusion improved mAP@50–95 in India and Czech compared with either model alone, but its overall score fell below both.
- YOLO26m had the highest overall score. YOLO26s was only **0.49 percentage points behind**, with faster inference on the same T4 GPU: **8.7 ms vs 24.1 ms per image**.

**Final model:** We selected the original **YOLO26s baseline at 640 pixels** for image and video inference. Its speed makes it the better fit for the demo, and combining both models did not improve overall validation mAP@50–95.

## Final Model Selection

We selected the original YOLO26s baseline at an input size of 640 pixels for image and video inference. It achieved **31.55% validation mAP@50–95**, within **0.49 percentage points** of YOLO26m, while running much faster on the same  GPU.

The cell below records the selected checkpoint and its SHA-256 fingerprint so the demo and final evaluation use the same weights.

In [38]:
from pathlib import Path
import hashlib
import json

drive_project = Path("/content/drive/MyDrive/road-damage-detection")
weights = drive_project / "training_runs/YOLO26s_baseline_model/weights/best.pt"

if not weights.is_file():
    raise FileNotFoundError(f"Check that Drive is mounted and this file exists:\n{weights}")

selection = {
    "model": "YOLO26s",
    "checkpoint": str(weights.relative_to(drive_project)),
    "sha256": hashlib.sha256(weights.read_bytes()).hexdigest(),
    "imgsz": 640,
    "validation_map50_95_pct": 31.55,
    "evaluation_ultralytics_version": "8.4.204",
}

record = drive_project / "final_model.json"

if record.exists():
    if json.loads(record.read_text()) != selection:
        raise RuntimeError("The saved selection differs. Review it before replacing it.")
    print("Final model selection already recorded.")
else:
    record.write_text(json.dumps(selection, indent=2) + "\n")
    print("Final model selection saved.")

print("Selected weights:", selection["checkpoint"])
print("SHA-256:", selection["sha256"])
print("Record:", record)

Final model selection saved.
Selected weights: training_runs/YOLO26s_baseline_model/weights/best.pt
SHA-256: 2304871205864929cfbb86739fe0867021e56de456d33b30f99c0f1fa6ab8f27
Record: /content/drive/MyDrive/road-damage-detection/final_model.json
